# Resolve a storage code explicitly.

Run the setup first, then each step in order. This notebook executes the same declarations, calls and assertions as the [script](04_storage_mapping.py). Outputs were captured by direct Python execution in a fresh process. No Jupyter kernel was used.


In [ ]:
!pip install -q aoa-action-machine

In [2]:
from typing import Annotated

from pydantic import Field

from aoa.action_machine.domain import AssociationOne, BaseCoreEntity, BaseEnumEntity, NoInverse, value
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entities import entities
from aoa.action_machine.intents.entity import entity
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.resources.base_storage import BaseStorage


class ShopDomain(BaseDomain):
    """Group the shop declarations."""

    name = "shop"
    description = "Shop model"


@entity(description="Currencies accepted by the shop", domain=ShopDomain)
class CurrencyEnumEntity(BaseEnumEntity):
    """Declare the shop's currency vocabulary."""

    symbol: str = Field(description="Currency symbol")

    EUR = value(code="eur", display_name="Euro", symbol="€")
    USD = value(code="usd", display_name="US dollar", symbol="$")


@entity(description="Price tag", domain=ShopDomain)
class PriceTagCoreEntity(BaseCoreEntity):
    """Associate an amount with its currency."""

    id: str = Field(description="Tag identifier")
    amount: float = Field(description="Price amount")
    currency: Annotated[AssociationOne[CurrencyEnumEntity], NoInverse()] = Field(description="Money unit")


@meta(description="Stores the shop entities")
@entities(
    CurrencyEnumEntity,
    PriceTagCoreEntity,
)
class ShopStorage(BaseStorage):
    """Hold the shop entities."""


def main() -> None:
    """Run this one learning experiment."""
    from aoa.action_machine.domain import build

    row = {"tag_key": "tag-1", "price": 9.99, "money_unit": "eur"}
    selected = CurrencyEnumEntity.by_code(row["money_unit"])
    tag = build(
        row,
        PriceTagCoreEntity,
        lambda fields, data: {
            fields.id: data["tag_key"],
            fields.amount: data["price"],
            fields.currency: AssociationOne(id=selected.code, entity=selected),
        },
    )
    print(tag.id, tag.currency.code)
    print("Shared value:", tag.currency.entity is CurrencyEnumEntity.EUR)


if __name__ == "__main__":
    main()

tag-1 eur
Shared value: True


## Load the integration API


## Run the next step


## Run the next step


## Run the next step


## Inspect the result


## Inspect the result
